# Notebook 5. The structure the game builds (Fig. 5)

Reproduces Fig. 5 and the degree-share columns of SI Tables S6 and S7. Reads no data and writes the panels of Fig. 5 to the folder `figs`. Run time about 8 minutes. `QUICK = True` in the first code cell shortens the runs for a test.

In [1]:
QUICK = False
T_RUN = 3000      # the run read by Fig. 4 and the copying analysis of Fig. S2
T_POP = 5000      # the four conditions, the hypothesis tests, the sweeps and Fig. 5; the same seeds,
                  # so the first 3,000 steps of these runs are the runs of Fig. 4

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from math import log2

plt.rcParams.update({'figure.dpi': 110, 'font.size': 8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.7,
                     'xtick.major.width': 0.7, 'ytick.major.width': 0.7,
                     'legend.frameon': False, 'font.family': 'DejaVu Sans'})

CEIL = log2(100)          # 6.6439 bits, the even-coupling ceiling at N = 100
BA_KAPPA = 10.10          # matched scale-free ensemble, control-entropy paper Table 1
BA_DH_DEG = 0.357         # its degree-weighted compression
BA_RDEG = -0.17           # its degree assortativity
N = 100
COL = {'baseline': '#7A7A7A', 'explore': '#2E7D4E', 'imitate': '#2E5090', 'both': '#A4161A'}
CONDS = {'baseline': dict(pr=0.0, pe=0.0), 'explore': dict(pr=0.0, pe=0.05),
         'imitate': dict(pr=0.3, pe=0.0), 'both': dict(pr=0.3, pe=0.05)}

def seeds(n):
    return list(range(n if not QUICK else max(2, n // 5)))

print('ceiling log2 N =', round(CEIL, 4), 'bits')

ceiling log2 N = 6.6439 bits


## Step 0. The model

In [2]:
import numpy as np
import networkx as nx

LOG2 = np.log(2.0)


# ----------------------------------------------------------------------
# measures (definitions of the second study)
# ----------------------------------------------------------------------
def control_entropy(v):
    """Control entropy in bits of a non-negative vector of shares (Eq. 11)."""
    v = np.asarray(v, float)
    tot = v.sum()
    if tot <= 0:
        return 0.0
    p = v / tot
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def eigen_shares(W):
    """Shares from the leading eigenvector of a symmetric coupling matrix."""
    vals, vecs = np.linalg.eigh(W)
    v = np.abs(vecs[:, -1])
    return v / v.sum()


def adjacency_measures(A, W):
    """Measures read from the unweighted structure, computed as in Table 1 of the
    control-entropy paper: eigenvector weighting on the largest connected component,
    degree assortativity on the whole graph."""
    G = nx.from_numpy_array(A.astype(int))
    deg = A.sum(axis=1)
    s = W.sum(axis=1)
    iu, ju = np.nonzero(np.triu(A, 1))

    def edge_corr(v):
        x, y = np.concatenate([v[iu], v[ju]]), np.concatenate([v[ju], v[iu]])
        return float(np.corrcoef(x, y)[0, 1]) if x.std() > 1e-12 else float('nan')

    r_deg, r_str = edge_corr(deg.astype(float)), edge_corr(s)
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    c = nx.eigenvector_centrality_numpy(G)
    v = np.abs(np.array([c[n] for n in G.nodes()]))
    return dict(H_adj=control_entropy(v), r_deg=float(r_deg), r_str=r_str, gc=G.number_of_nodes())


# ----------------------------------------------------------------------
# the five rules, one function each
# ----------------------------------------------------------------------
def ring_lattice(N, k):
    """Regular ring lattice: every element coupled to its k nearest neighbours."""
    return nx.to_numpy_array(nx.watts_strogatz_graph(N, k, 0.0), dtype=bool)


def softmax_allocation(A, beta, shat):
    """Eq 1. Element i splits one unit of coupling over its partners j with weights
    exp(beta_i * shat_j). beta_i = 0 gives even coupling."""
    L = np.where(A, beta[:, None] * shat[None, :], -np.inf)
    L = L - L.max(axis=1, keepdims=True)
    E = np.where(A, np.exp(L), 0.0)
    return E / E.sum(axis=1, keepdims=True)


def commitment(phi0, phi1, C, fm):
    """Eq 2. Commitment of an element, in the linear form of the published model,
    f = phi0 + phi1 * C, truncated to [0, 1 - fm]."""
    return np.clip(phi0 + phi1 * C, 0.0, 1.0 - fm)


def protection(f, c, pmax, chalf):
    """Eq 8. Saturating protection bought by committed capability f * c."""
    x = f * c
    return np.where(x > 0, pmax / (1.0 + chalf / np.maximum(x, 1e-12)), 0.0)


def fermi(dc, s):
    """Eq 10. Probability of copying a partner whose capability is higher by dc."""
    return 1.0 / (1.0 + np.exp(-s * dc))


def logistic(z, g, theta):
    """Response function of an element, used inside the spread (Eq 5)."""
    return 1.0 / (1.0 + np.exp(-g * (z - theta)))


class ElementModel:
    """Network-agent model of element-level exploration and imitation.

    Each element holds
      beta_i        direction of coupling (0 = even coupling, > 0 = toward stronger partners)
      phi0_i, phi1_i commitment strategy, giving f_i = phi0_i + phi1_i * C_i
      c_i           capability, the accumulated stimulation it compares with partners
    Each coupling (i, j) holds a weight w_ij.
    """

    def __init__(self, N=100, k=6, seed=0,
                 pr=0.3, pe=0.05, sigma=0.1, sigma_f=0.01, s=2.0,   # strategy dynamics
                 M=10, rho=0.5, nE=5, switch=None, psi=0.3,        # feedback, two channels
                 gamma=1.0, pn=0.02, pl=0.3, lam=1.0, g=4.0, theta=1.0,   # seeding and spread
                 fm=0.1, pmax=1.0, chalf=0.1, f0=0.1,              # capability and protection
                 eta=0.05, delta=0.01, w0=1.0,                     # consolidation
                 pw=2.0, kmin=2, loss=1.0,                         # pruning and its cost
                 spread=True, two_part=True):
        self.p = dict(N=N, k=k, pr=pr, pe=pe, sigma=sigma, sigma_f=sigma_f, s=s, M=M, rho=rho,
                      nE=nE, switch=switch, psi=psi, gamma=gamma, pn=pn, pl=pl, lam=lam, g=g,
                      theta=theta, fm=fm, pmax=pmax, chalf=chalf, f0=f0, eta=eta, delta=delta,
                      w0=w0, pw=pw, kmin=kmin, loss=loss, spread=spread, two_part=two_part)
        self.rng = np.random.default_rng(seed)
        self.N, self.k = N, k
        self.A = ring_lattice(N, k)
        self.W = self.A.astype(float) * w0
        self.beta = np.zeros(N)
        self.phi0 = np.full(N, f0)
        self.phi1 = np.zeros(N)
        self.c = np.ones(N)
        self.x = np.zeros(N)          # activation of the previous step
        self.v = np.ones(N) / np.sqrt(N)   # leading eigenvector, kept warm
        self.t = 0
        self.n_act = 0.0
        self.n_rounds = 0.0
        self.set_essential(self.rng.choice(N, nE, replace=False))

    # ---------- state helpers ----------
    def set_essential(self, E):
        self.E = np.asarray(E)
        self.isE = np.zeros(self.N, bool)
        self.isE[self.E] = True

    def strengths(self):
        return self.W.sum(axis=1)

    def shat(self):
        s = self.strengths()
        return s / s.max()

    def centrality(self, iters=8):
        """Leading eigenvector of W by power iteration, warm started, scaled to a
        maximum of one. This is the C of the commitment rule (Eq 2)."""
        v = self.v
        for _ in range(iters):
            v = self.W @ v
            n = np.linalg.norm(v)
            if n < 1e-12:
                v = np.ones(self.N) / np.sqrt(self.N)
                break
            v = v / n
        self.v = np.abs(v)
        return self.v / max(self.v.max(), 1e-12)

    def allocation(self):
        return softmax_allocation(self.A, self.beta, self.shat())

    # ---------- one step ----------
    def feedback(self):
        """Eq 3. Two channels of information.
        Process feedback is intrinsic: every element receives afferent input in
        proportion to its own activation in the previous step.
        Result feedback is augmented: M packets, each landing on a task-essential
        element with probability rho and on any element otherwise."""
        p, rng = self.p, self.rng
        u = p['psi'] * self.x.copy()
        if p['M'] > 0:
            to_E = rng.random(p['M']) < p['rho']
            tgt = np.where(to_E, rng.choice(self.E, p['M']), rng.integers(0, self.N, p['M']))
            u = u + np.bincount(tgt, minlength=self.N).astype(float)
        return u

    def spread(self, u, a):
        """Eq 4 and Eq 5. Stimulus is seeded where information arrives and then
        travels along the couplings.
        Seeding: element i is stimulated with probability 1 - (1 - p_n) exp(-gamma u_i),
        so an element fires spontaneously at rate p_n and more often when information
        reaches it.
        Spread: an element already stimulated passes activation to partner j with
        the coupling it allocates, and j responds with probability
        pl * logistic(g (d_j - theta)). Each element activates at most once per step.
        The cascade is run to completion inside the step."""
        p, rng, N = self.p, self.rng, self.N
        act = rng.random(N) < (1.0 - (1.0 - p['pn']) * np.exp(-p['gamma'] * u))   # Eq 4
        if not p['spread']:
            self.n_act, self.n_rounds = float(act.sum()), 0.0
            return act
        frontier = act.copy()
        rounds = 0
        while frontier.any() and rounds < 20:
            # drive reaching each candidate through the allocation of active elements
            d = p['lam'] * self.k * (frontier @ a)
            prob = p['pl'] * logistic(d, p['g'], p['theta'])       # Eq 5
            new = (rng.random(N) < prob) & ~act
            act |= new
            frontier = new
            rounds += 1
        self.n_act, self.n_rounds = float(act.sum()), float(rounds)
        return act

    def step(self):
        p, rng, N = self.p, self.rng, self.N
        if p['switch'] and self.t > 0 and self.t % p['switch'] == 0:
            self.set_essential(rng.choice(N, p['nE'], replace=False))

        C = self.centrality()
        a = self.allocation()                                     # Eq 1
        f = commitment(self.phi0, self.phi1, C, p['fm']) if p['two_part'] else np.zeros(N)  # Eq 2
        u = self.feedback()                                       # Eq 3
        act = self.spread(u, a)                                   # Eq 4, Eq 5
        x = act.astype(float)
        self.x = x

        # Eq 6. Payoff is the stimulation the element gathers. It reaches partners
        # through its own allocation and it integrates in proportion to the coupling
        # it holds, so both the direction and the amount of coupling enter.
        s = self.strengths()
        pay = (s / s.mean()) * (a @ x)
        self.c = pay + np.clip(1.0 - p['fm'] - f, 0.0, None) * self.c   # Eq 7

        # Eq 8 and Eq 9. Commitment protects a coupling against decay.
        P = protection(f, self.c, p['pmax'], p['chalf'])
        Pij = 0.5 * (P[:, None] + P[None, :])
        abar = 0.5 * (a + a.T)
        self.W = np.where(self.A,
                          p['w0'] + (1 - p['delta'] * (1 - Pij)) * (self.W - p['w0'])
                          + p['eta'] * self.k * abar * np.outer(x, x), 0.0)

        # Eq S1. Long-term change: a coupling falls out of use, and unless the
        # capability committed to it holds it, it is lost and replaced elsewhere.
        n_rw = int(p['pw']) + (1 if rng.random() < p['pw'] - int(p['pw']) else 0)
        for _ in range(n_rw):
            self._rewire(P)

        # Eq 10. Imitation along a coupling, synchronous, two independent passes.
        b_old, f0_old, f1_old, c_old = self.beta.copy(), self.phi0.copy(), self.phi1.copy(), self.c.copy()
        for i in np.flatnonzero(rng.random(N) < p['pr']):
            nb = np.flatnonzero(self.A[i])
            j = nb[rng.integers(nb.size)]
            if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                self.beta[i] = b_old[j]
        if p['two_part']:
            for i in np.flatnonzero(rng.random(N) < p['pr']):
                nb = np.flatnonzero(self.A[i])
                j = nb[rng.integers(nb.size)]
                if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                    self.phi0[i], self.phi1[i] = f0_old[j], f1_old[j]

        # Eq S2. Exploration of both parts of the strategy.
        ex = rng.random(N) < p['pe']
        self.beta[ex] += rng.normal(0.0, p['sigma'], ex.sum())
        if p['two_part']:
            ex0 = rng.random(N) < p['pe']
            self.phi0[ex0] += rng.normal(0.0, p['sigma_f'], ex0.sum())
            ex1 = rng.random(N) < p['pe']
            self.phi1[ex1] += rng.normal(0.0, p['sigma_f'], ex1.sum())
        self.t += 1
        return x

    def _rewire(self, P):
        """One element drops its weakest coupling, unless protection holds it, and
        attaches to a new partner chosen with exp(beta_i * shat_j)."""
        p, rng = self.p, self.rng
        i = rng.integers(self.N)
        deg = self.A.sum(axis=1)
        nb = np.flatnonzero(self.A[i])
        nb = nb[deg[nb] > p['kmin']]
        if deg[i] <= p['kmin'] or nb.size == 0:
            return
        j = nb[np.argmin(self.W[i, nb])]
        if rng.random() < 0.5 * (P[i] + P[j]):     # the coupling is held
            return
        # the route is lost, and with it the capability the two elements had built
        # through it. With loss = 1 this is the capability loss of the published model.
        self.c[i] *= (1.0 - p['loss'])
        self.c[j] *= (1.0 - p['loss'])
        cand = np.flatnonzero(~self.A[i])
        cand = cand[cand != i]
        if cand.size == 0:
            return
        z = self.beta[i] * self.shat()[cand]
        w = np.exp(z - z.max())
        new = rng.choice(cand, p=w / w.sum())
        self.A[i, j] = self.A[j, i] = False
        self.W[i, j] = self.W[j, i] = 0.0
        self.A[i, new] = self.A[new, i] = True
        self.W[i, new] = self.W[new, i] = p['w0']

    # ---------- measures ----------
    def measures(self, adj=False):
        s = self.strengths()
        deg = self.A.sum(axis=1)
        top = np.argsort(s)[-len(self.E):]
        nbE = self.A[self.E].any(axis=0) & ~self.isE
        tot = s.sum()
        C = self.centrality()
        f = commitment(self.phi0, self.phi1, C, self.p['fm'])
        return dict(H_str=control_entropy(s), H_eig=control_entropy(eigen_shares(self.W)),
                    H_deg=control_entropy(deg), kappa=(deg ** 2).mean() / deg.mean(),
                    beta_mean=self.beta.mean(), beta_var=self.beta.var(),
                    f_mean=float(f.mean()), phi0_mean=float(self.phi0.mean()),
                    phi1_mean=float(self.phi1.mean()), c_mean=float(self.c.mean()),
                    act=self.n_act / self.N, rounds=self.n_rounds,
                    hub_align=self.isE[top].mean(),
                    share_E=s[self.isE].sum() / tot, share_nbE=s[nbE].sum() / tot,
                    **(adjacency_measures(self.A, self.W) if adj else {}))

    def run(self, T, every=25, adj=False):
        rec = {}
        for step in range(T):
            self.step()
            if step % every == 0 or step == T - 1:
                m = self.measures(adj=adj)
                m['t'] = self.t
                for key, v in m.items():
                    rec.setdefault(key, []).append(v)
        return {key: np.array(v) for key, v in rec.items()}


def replicate(T, seeds, adj=True, **kw):
    """Run the model for several seeds and return the final measures."""
    out = []
    for sd in seeds:
        m = ElementModel(seed=sd, **kw)
        for _ in range(T):
            m.step()
        out.append(m.measures(adj=adj))
    return {key: np.array([o[key] for o in out]) for key in out[0]}

In [3]:
from scipy.stats import mannwhitneyu

def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > y) for x in a for y in b)
    lt = sum((x < y) for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))

def boot_median(a, reps=5000, seed=0):
    rng2 = np.random.default_rng(seed)
    a = np.asarray(a)
    d = [np.median(rng2.choice(a, len(a))) for _ in range(reps)]
    return np.percentile(d, [2.5, 97.5])

def boot_diff(a, b, reps=5000, seed=0):
    rng2 = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    d = [np.median(rng2.choice(a, len(a))) - np.median(rng2.choice(b, len(b))) for _ in range(reps)]
    return np.percentile(d, [2.5, 97.5])

def summary(tag, v, fmt='.3f'):
    v = np.asarray(v, float); lo, hi = boot_median(v)
    print(f'{tag}: median {np.median(v):{fmt}} [{lo:{fmt}}, {hi:{fmt}}], '
          f'mean {v.mean():{fmt}}, sd {v.std(ddof=1):{fmt}}, range {v.min():{fmt}} to {v.max():{fmt}}')

def compare(tag, a, b, unit=''):
    u, p = mannwhitneyu(a, b, alternative='two-sided')
    d = cliffs_delta(a, b)
    lo, hi = boot_diff(a, b)
    diff = np.median(a) - np.median(b)
    print(f'{tag}: U = {u:.1f}, p = {p:.4f}, Cliff delta = {d:+.2f}, '
          f'median difference = {diff:+.3f}{unit} [{lo:+.3f}, {hi:+.3f}]')
    return dict(U=u, p=p, d=d, diff=diff, lo=lo, hi=hi)

def star(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'

## Step 1. Four conditions followed from the lattice

In [4]:
EVERY = 25
KEYS = ('H_str', 'H_deg', 'H_eig', 'kappa', 'beta_mean', 'beta_var', 'f_mean', 'share_E')

def read_quietly(m):
    v = m.v.copy(); mm = m.measures(adj=False); m.v = v
    return mm

def follow(name, seed, T=None):
    T = T or T_POP
    m = ElementModel(seed=seed, **CONDS[name])
    rec = {k: [] for k in KEYS}; rec['t'] = []
    def keep():
        mm = read_quietly(m)
        for k in KEYS:
            rec[k].append(mm[k])
        rec['t'].append(m.t)
    keep()
    snaps = {0: dict(W=m.W.copy(), beta=m.beta.copy(), isE=m.isE.copy())}
    pred, meas = np.zeros(T), np.zeros(T)
    pr, s = m.p['pr'], m.p['s']
    for t in range(1, T + 1):
        b_prev = m.beta.copy()
        m.step()
        if name == 'both':
            A = m.A.astype(bool)
            D = fermi(m.c[None, :] - m.c[:, None], s) * (b_prev[None, :] - b_prev[:, None]) * A
            pred[t - 1] = pr / m.N * np.sum(D.sum(1) / A.sum(1))
            meas[t - 1] = m.beta.mean() - b_prev.mean()
        if t % EVERY == 0:
            keep()
    snaps[T] = dict(W=m.W.copy(), beta=m.beta.copy(), isE=m.isE.copy())
    return dict(seed=seed, rec={k: np.array(v) for k, v in rec.items()}, pred=pred, meas=meas, snaps=snaps)

NS = seeds(30)
RUNS = {name: [follow(name, sd) for sd in NS] for name in CONDS}
T = RUNS['both'][0]['rec']['t']; TMAX = int(T[-1])
print(f'{len(NS)} seeds in each of the four conditions, {T_POP} steps, read every {EVERY} steps')

plain = ElementModel(seed=0, **CONDS['both'])
for _ in range(T_POP):
    plain.step()
CHECK_PLAIN = bool(np.array_equal(plain.W, RUNS['both'][0]['snaps'][TMAX]['W']))
print('check, a followed run equals the same seed run straight through:', CHECK_PLAIN)

30 seeds in each of the four conditions, 5000 steps, read every 25 steps
check, a followed run equals the same seed run straight through: True


## Step 2. The structure at the end, and the run drawn in panel C

In [5]:
def stack(name, key):
    return np.array([r['rec'][key] for r in RUNS[name]])

def degrees(name):
    return [(r['snaps'][TMAX]['W'] > 0).sum(1) for r in RUNS[name]]

END = {}
print('condition      kappa    compression, strength   compression, degree   largest degree   share on essential')
for name in CONDS:
    k = stack(name, 'kappa')[:, -1]; ds = CEIL - stack(name, 'H_str')[:, -1]; dd = CEIL - stack(name, 'H_deg')[:, -1]
    mx = np.array([d.max() for d in degrees(name)]); se = stack(name, 'share_E')[:, -1]
    END[name] = dict(kappa=k, dH_str=ds, dH_deg=dd, maxdeg=mx, share=se)
    print(f'{name:<12} {np.median(k):6.2f}    {np.median(ds):.3f}                   {np.median(dd):.3f}'
          f'                 {np.median(mx):5.1f}            {np.median(se):.3f}')
rest = lambda key: np.concatenate([END[n][key] for n in ('baseline', 'explore', 'imitate')])
print(f'the other three pooled: compression on degree shares median {np.median(rest("dH_deg")):.3f}, '
      f'largest degree median {np.median(rest("maxdeg")):.1f}')
pool = {n: np.concatenate(degrees(n)) for n in CONDS}
for n in CONDS:
    print(f'{n:<12} pooled degrees: largest {pool[n].max()}, share of elements with degree 20 or more '
          f'{(pool[n] >= 20).mean():.3f}')

kap = END['both']['kappa']
REP = RUNS['both'][int(np.argmin(np.abs(kap - np.median(kap))))]
W0, W1 = REP['snaps'][0]['W'], REP['snaps'][TMAX]['W']
def degree_compression(W):
    q = (W > 0).sum(1) / (W > 0).sum()
    return CEIL - float(-(q[q > 0] * np.log2(q[q > 0])).sum())
d1 = (W1 > 0).sum(1); s1 = W1.sum(1)
REP_VAL = dict(seed=REP['seed'], kappa=REP['rec']['kappa'][-1], maxdeg=int(d1.max()),
               beta=REP['snaps'][TMAX]['beta'].mean(), dH0=degree_compression(W0), dH1=degree_compression(W1),
               top5=int(REP['snaps'][0]['isE'][np.argsort(-s1)[:5]].sum()), deg2=int((d1 == d1.min()).sum()),
               dmin=int(d1.min()))
print('compression on degree shares, medians with 95 percent bootstrap intervals')
DHDEG_CI = {}
for name in CONDS:
    DHDEG_CI[name] = boot_median(END[name]['dH_deg'])
    print(f'  {name:<12} {np.median(END[name]["dH_deg"]):.3f} [{DHDEG_CI[name][0]:.3f}, {DHDEG_CI[name][1]:.3f}]')
T_DEG_EXP = compare('compression on degree shares, both vs exploration only', END['both']['dH_deg'],
                    END['explore']['dH_deg'], ' bits')
T_DEG_BASE = compare('compression on degree shares, both vs baseline', END['both']['dH_deg'],
                     END['baseline']['dH_deg'], ' bits')

print(f'panel C: seed {REP_VAL["seed"]}, kappa {REP_VAL["kappa"]:.2f} against the median {np.median(kap):.2f}, '
      f'largest degree {REP_VAL["maxdeg"]}, smallest degree {REP_VAL["dmin"]} ({REP_VAL["deg2"]} elements), '
      f'mean preference {REP_VAL["beta"]:+.2f}, compression on degree shares {REP_VAL["dH0"]:.2f} at the start and '
      f'{REP_VAL["dH1"]:.2f} at the end, task-essential elements among the five strongest {REP_VAL["top5"]} of 5')

condition      kappa    compression, strength   compression, degree   largest degree   share on essential
baseline       9.73    0.631                   0.367                  24.0            0.230
explore       10.27    0.881                   0.402                  28.5            0.233
imitate        9.79    0.648                   0.371                  24.0            0.235
both          21.04    1.036                   0.709                  84.5            0.391
the other three pooled: compression on degree shares median 0.379, largest degree median 25.0
baseline     pooled degrees: largest 32, share of elements with degree 20 or more 0.025
explore      pooled degrees: largest 37, share of elements with degree 20 or more 0.026
imitate      pooled degrees: largest 29, share of elements with degree 20 or more 0.023
both         pooled degrees: largest 99, share of elements with degree 20 or more 0.035
compression on degree shares, medians with 95 percent bootstrap intervals
  base

## Step 3. Variation turns into direction: Eq. 15 against the runs

In [6]:
P_SUM = np.array([r['pred'].sum() for r in RUNS['both']])
M_SUM = np.array([r['meas'].sum() for r in RUNS['both']])
DIFF = P_SUM - M_SUM
SE = DIFF.std(ddof=1) / np.sqrt(len(DIFF))
bm = stack('both', 'beta_mean').mean(0)
t05 = int(T[np.argmax(bm >= 0.5)]) if (bm >= 0.5).any() else None
fm = stack('both', 'f_mean').mean(0)
t_f = int(T[np.argmax(fm < 0.01)]) if (fm < 0.01).any() else None
r_runs = np.corrcoef(P_SUM, M_SUM)[0, 1]
print(f'mean over {len(DIFF)} runs: Eq. 15 summed {P_SUM.mean():+.3f}, measured {M_SUM.mean():+.3f}, '
      f'difference {DIFF.mean():+.3f} (standard error {SE:.3f}), correlation over runs {r_runs:+.2f}')
print(f'mean preference first at 0.5 or above at step {t05}; mean commitment first below 0.01 at step {t_f}')
CHECK_EQ15 = bool(abs(DIFF.mean()) < 2 * SE + 0.05)
print('check, Eq. 15 summed within two standard errors of the measured change:', CHECK_EQ15)

mean over 30 runs: Eq. 15 summed +1.956, measured +1.739, difference +0.216 (standard error 0.185), correlation over runs +0.71
mean preference first at 0.5 or above at step 1325; mean commitment first below 0.01 at step 350
check, Eq. 15 summed within two standard errors of the measured change: True


## Step 4. Figure 5

In [7]:
import os
from matplotlib.transforms import Bbox

PANEL_DIR = os.environ.get('FIG_DIR', 'figs')


def save_panels(fig, stem, regions, hide=()):
    # every region of the drawn figure saved as its own panel file, PNG at 300 dpi and PDF and SVG with editable
    # text; regions maps a panel name to its box (x0, y0, x1, y1) in inches from the lower left corner of the
    # figure, and the artists in hide, which join the panels, are left out of the panel files
    os.makedirs(PANEL_DIR, exist_ok=True)
    for a in hide:
        a.set_visible(False)
    with plt.rc_context({'pdf.fonttype': 42, 'svg.fonttype': 'none'}):
        for name, box in regions.items():
            path = os.path.join(PANEL_DIR, f'{stem}{name}')
            kw = dict(bbox_inches=Bbox.from_extents(*box), facecolor='white')
            fig.savefig(path + '.png', dpi=300, metadata={'Software': None}, **kw)
            fig.savefig(path + '.pdf', metadata={'Creator': None, 'Producer': None, 'CreationDate': None}, **kw)
            fig.savefig(path + '.svg', metadata={'Creator': None, 'Date': None}, **kw)
            svg = open(path + '.svg', encoding='utf-8').read().replace('id="matplotlib.', 'id="')
            open(path + '.svg', 'w', encoding='utf-8').write(svg)
    for a in hide:
        a.set_visible(True)
    print('panel files:', ', '.join(f'{stem}{n}' for n in regions))

In [8]:
import matplotlib as mpl
import matplotlib.patheffects
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.collections import LineCollection
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.labelsize': 6.4, 'xtick.labelsize': 6.0,
                     'ytick.labelsize': 6.0, 'legend.fontsize': 6.0, 'axes.linewidth': 0.6,
                     'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.4,
                     'ytick.major.size': 0, 'mathtext.fontset': 'dejavusans'})
FIG_W = 6.93
PW_T, PH_T = FIG_W / 4, 2.02
PW_B, PH_B = FIG_W / 3, 1.52
GAP = 0.06
BOX_L = (0.42, 0.33, 1.62, 0.94)
BOX_R = (PW_B - 0.42 - 1.62, 0.33, 1.62, 0.94)
BOX_2 = (0.42, 0.33, PW_B - 0.84, 0.94)
INK, NEUTRAL, LINK = '#1A1A1A', '#37474F', '#C3CBD0'
BLUE, RED, GOLD, GREEN, GREY, NAVY = '#2E5090', '#A4161A', '#C9A227', '#2E7D4E', '#8A8A8A', '#1F3F7A'
_HOT = plt.get_cmap('hot_r')(np.linspace(0.06, 0.94, 256))
_COOL = plt.get_cmap('Blues')(np.linspace(0.92, 0.06, 256))
CMAP = LinearSegmentedColormap.from_list('hot_div', np.vstack([_COOL, _HOT]))
VLIM = 2.7157337518864986
CNORM = Normalize(-VLIM, VLIM)


def colour(w, n):
    b = np.log2(np.asarray(w, float) * n)
    return CMAP(CNORM(np.where(np.abs(b) < 1e-9, 0.0, b)))


def axes_at(fig, ox, oy, box):
    W, H = fig.get_size_inches(); l, b, w, h = box
    return fig.add_axes([(ox + l) / W, (oy + b) / H, w / W, h / H])


def heading(fig, ox, oy, pw, ph, letter, title, sub=None):
    W, H = fig.get_size_inches(); y = (oy + ph - 0.19) / H
    fig.text((ox + 0.05) / W, y, letter, fontsize=9.4, fontweight='bold', color='k', ha='left', va='bottom')
    fig.text((ox + pw / 2) / W, y, title, fontsize=7.9, color='k', ha='center', va='bottom')
    if sub:
        fig.text((ox + pw / 2) / W, y - 0.02 / H, sub, fontsize=6.2, color='k', ha='center', va='top')


def frame(ax, xt, yt):
    ax.set_xlim(xt[0], xt[-1]); ax.set_ylim(yt[0], yt[-1]); ax.set_xticks(xt); ax.set_yticks(yt)
    fmt = mpl.ticker.FuncFormatter(lambda v, _: f'{v:,g}')
    ax.xaxis.set_major_formatter(fmt); ax.yaxis.set_major_formatter(fmt)
    ax.xaxis.set_minor_locator(mpl.ticker.NullLocator()); ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
    for s_ in ('left', 'right'):
        ax.spines[s_].set_visible(False)
    for s_ in ('top', 'bottom'):
        ax.spines[s_].set_visible(True); ax.spines[s_].set_linewidth(0.7)
    ax.grid(True, axis='y', alpha=0.40, linestyle=':', linewidth=0.5, color='gray'); ax.set_axisbelow(True)
    ax.tick_params(axis='y', length=0, pad=1.4)
    ax.tick_params(axis='x', length=2.4, width=0.6, direction='out', pad=1.4)

MOMENTS = (0, TMAX)
XT = [0, TMAX // 2, TMAX]


NET, LIM = 1.46, 1.13
PT = NET / (2 * LIM) * 72                                            # points per data unit in the network panels


def node_size(d):
    return 5.0 + 190.0 * ((d - 1) / (N - 2)) ** 1.15


def apart(xy, rad, movable, pinned, iters=400):
    '''Push overlapping glyphs apart. Only elements drawn inward move, the main hub stays at the center.'''
    xy = xy.copy()
    for _ in range(iters):
        moved = False
        for a in movable:
            for b in range(N):
                if a == b:
                    continue
                v = xy[a] - xy[b]; dist = np.hypot(*v); need = 1.18 * (rad[a] + rad[b])
                if dist < need:
                    u = v / dist if dist > 1e-9 else np.array([1.0, 0.0])
                    if b in movable and b != pinned:
                        xy[a] += 0.5 * (need - dist) * u; xy[b] -= 0.5 * (need - dist) * u
                    else:
                        xy[a] += (need - dist) * u
                    moved = True
        if not moved:
            break
    return xy


def wavy(ax, p0, p1, amp=0.035, waves=3.5, lw=0.8, halo=False, label='u', side='right', ms=2.6, fs=7.0):
    '''The stimulus u as in Fig. 4B: a dot, a wavy line and an arrowhead. p0 and p1 in data units of an axes
    whose two data units have the same length on the page.'''
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    v = p1 - p0; L = np.hypot(*v); e = v / L; n = np.array([-e[1], e[0]])
    tt = np.linspace(0, 0.78, 160)
    pts = p0[None] + (tt * L)[:, None] * e[None] + (amp * np.sin(2 * np.pi * waves * tt / 0.78))[:, None] * n[None]
    pe = [mpl.patheffects.withStroke(linewidth=lw + 2.2, foreground='white')] if halo else None
    ax.plot(pts[:, 0], pts[:, 1], color=INK, lw=lw, solid_capstyle='round', zorder=7, path_effects=pe,
            clip_on=False)
    ar = FancyArrowPatch(p0 + 0.76 * v, p1, arrowstyle='-|>', mutation_scale=6.5, color=INK, lw=lw, zorder=7,
                         shrinkA=0, shrinkB=0, clip_on=False)
    if halo:
        ar.set_path_effects([mpl.patheffects.withStroke(linewidth=lw + 2.2, foreground='white')])
    ax.add_patch(ar)
    ax.plot([p0[0]], [p0[1]], 'o', ms=ms, color=INK, zorder=8, clip_on=False,
            path_effects=[mpl.patheffects.withStroke(linewidth=1.6, foreground='white')] if halo else None)
    off = {'right': (3.6, 0.6), 'left': (-3.6, 0.6), 'above': (0.0, 3.4)}[side]
    ax.annotate(label, xy=p0, xytext=off, textcoords='offset points', fontsize=fs, style='italic', color=INK,
                ha={'right': 'left', 'left': 'right', 'above': 'center'}[side],
                va='bottom' if side == 'above' else 'center', zorder=8, annotation_clip=False)


DEG_RINGS = (3, 10, 30)


def radius(d, dmax, dmin=2.0):
    '''Radius of an element at the end of the run, on a log scale of degree: the least coupled stay on the
    ring of the lattice, the largest hub sits at the center.'''
    return np.clip(np.log(dmax / np.maximum(d, dmin)) / np.log(dmax / dmin), 0, 1)


def draw_net(fig, ox, oy, t, letter, title, sub):
    '''The elements keep their places on the ring of the lattice. At the end each one is drawn inward by its
    degree on a log scale, so the hubs move to the center and the many weakly coupled elements stay outside.'''
    W = REP['snaps'][t]['W']; A = W > 0; d = A.sum(1).astype(float); s = W.sum(1); w = s / s.sum()
    th = np.pi / 2 - 2 * np.pi * np.arange(N) / N
    if t == 0:
        r = np.ones(N); size = np.full(N, 6.0)
    else:
        r = radius(d, d.max())
        size = node_size(d)
    xy = np.c_[r * np.cos(th), r * np.sin(th)]
    rad = np.sqrt(size) / 2 / PT
    if t:
        hub = int(np.argmax(d))
        xy = apart(xy, rad, [i for i in range(N) if d[i] > 4 and i != hub], hub)
    ax = axes_at(fig, ox, oy, ((PW_T - NET) / 2, 0.25, NET, NET))
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect('equal'); ax.axis('off')
    iu = np.array(np.where(np.triu(A))).T
    if t == 0:                                    # lattice links as arcs inside the ring, deeper for longer ones
        segs = []
        for a, b in iu:
            m = min((b - a) % N, (a - b) % N)
            ta, tb = th[a], th[a] - 2 * np.pi * m / N * (1 if (b - a) % N == m else -1)
            ctrl = (1 - 0.075 * m) * np.array([np.cos((ta + tb) / 2), np.sin((ta + tb) / 2)])
            u = np.linspace(0, 1, 14)[:, None]
            segs.append((1 - u) ** 2 * xy[a] + 2 * u * (1 - u) * ctrl + u ** 2 * xy[b])
        ax.add_collection(LineCollection(segs, colors='#8E9AA2', linewidths=0.42, alpha=0.9, zorder=1))
    else:                                         # straight couplings, darker and wider where consolidated
        for kk in DEG_RINGS:                      # degree read off the radius
            rr = radius(np.array([kk]), d.max())[0]
            ax.add_patch(mpl.patches.Circle((0, 0), rr, fill=False, ec='#9AA6AD', lw=0.45,
                                            ls=(0, (1.2, 1.6)), zorder=0))
            ax.text(rr * np.cos(np.radians(-58)), rr * np.sin(np.radians(-58)), f'{kk}', fontsize=5.0,
                    color='#5F6B72', ha='center', va='center', zorder=5,
                    bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=0.9))
        ww = W[iu[:, 0], iu[:, 1]]; x = np.sqrt((ww - ww.min()) / (ww.max() - ww.min()))
        o = np.argsort(ww)
        ax.add_collection(LineCollection(xy[iu[o]], colors=plt.get_cmap('Greys')(0.22 + 0.62 * x[o]),
                                         linewidths=0.18 + 1.25 * x[o], alpha=0.85, zorder=1))
    o = np.argsort(d, kind='stable')
    ax.scatter(xy[o, 0], xy[o, 1], s=size[o], c=colour(w, N)[o], edgecolors=NEUTRAL, linewidths=0.3, zorder=3)
    e = np.where(REP['snaps'][0]['isE'])[0]
    ax.scatter(xy[e, 0], xy[e, 1], s=(np.sqrt(size[e]) + 3.4) ** 2, facecolors='none', edgecolors=INK,
               linewidths=0.75, zorder=4)
    # the stimulus u, which lands mostly on the task-essential elements (Eq. 3)
    p0 = np.array([-1.04, 1.06])
    tgt = e[np.argmin(np.hypot(*(xy[e] - p0).T))]
    v = xy[tgt] - p0; L = np.hypot(*v)
    p1 = xy[tgt] - v / L * (rad[tgt] + 3.4 / PT + 0.03)
    wavy(ax, p0, p1, amp=0.032, waves=3.0 if t == 0 else 4.5, halo=t != 0)
    if t == 0:                                    # the key sits in the empty middle of the ring
        ax.scatter([-0.42], [0.0], s=(np.sqrt(6.0) + 3.4) ** 2, facecolors='none', edgecolors=INK, linewidths=0.75)
        ax.scatter([-0.42], [0.0], s=6.0, c=colour([1 / N], N), edgecolors=NEUTRAL, linewidths=0.3)
        ax.text(-0.31, 0.0, 'task-essential\nelement', fontsize=5.8, ha='left', va='center', linespacing=0.95)
    heading(fig, ox, oy, PW_T, PH_T, letter, title, sub)
    # the micro state and the macro measure at that moment
    bbar = REP['snaps'][t]['beta'].mean()
    q = A.sum(1) / A.sum()
    dH = CEIL - (-(q[q > 0] * np.log2(q[q > 0])).sum())
    W_, H_ = fig.get_size_inches()
    fig.text((ox + PW_T / 2) / W_, (oy + 0.12) / H_,
             (rf'$\bar{{\beta}}$ = {bbar:+.2f}      $\Delta H$ = {dH:.2f} bits').replace('-', '\u2212'),
             fontsize=6.6, color='k', ha='center', va='center')



def draw_game(fig, ox, oy, letter, pw):
    '''The game of one step in three stages, schematic, values chosen to show the rules. Players are the
    elements, the strategy is the direction preference, the payoff is the stimulus gathered and kept as
    capability, and every element revises at once along its couplings. Three elements are enough to show it:
    i and j as in Fig. 4, and l, the strong partner that the stimulus u reaches. After variation the three
    still earn alike and lie on one line. After the payoff the four corners of the square are the strategies
    and capabilities: i and l share a strategy, j and l share a capability, so copying l changes nothing and
    only copying j moves the mean strategy.'''
    W_, H_ = fig.get_size_inches()
    heading(fig, ox, oy, pw, PH_T, letter, 'An element-level evolutionary game',
            'simultaneous, along the couplings, every step')
    bL, bl, bR, cL, cH = 0.10, 0.68, 1.60, 1.00, 2.50
    LINKS = [('i', 'k'), ('k', 'j'), ('i', 'j')]
    SHOW = {'i': 'i', 'j': 'j', 'k': 'l'}       # the partner is l, as in Fig. 4, since k is the degree
    sw, sh = 1.00, 0.99
    gap = (pw - 0.16 - 3 * sw) / 2
    x0, y0 = ox + 0.08, oy + 0.56
    XL, YL = (-0.75, 2.15), (0.30, 0.30 + 2.90 * sh / sw)
    xpi, ypi = (XL[1] - XL[0]) / sw, (YL[1] - YL[0]) / sh
    R_ = 0.085 * xpi                                                   # glyph radius in plane units
    cM = (YL[0] + YL[1]) / 2
    S1 = {'i': (bL, cM), 'k': (bl, cM), 'j': (bR, cM)}                # after variation, on one line
    S2 = {'i': (bL, cL), 'k': (bL, cH), 'j': (bR, cH)}                # after payoff, a square with the copy
    titles = ['1  Variation', '2  Payoff', '3  Copy']
    rules = [r'$\beta_i \rightarrow \beta_i + \varepsilon_i,\ \ \varepsilon_i \sim N(0, \sigma^2)$',
             r'$\partial\pi_i/\partial\beta_i \propto \mathrm{Cov}_{a_i}(\hat{s}, x)$',
             r'$\varphi(c_j - c_i) = 1/[1 + e^{-s(c_j - c_i)}]$']

    def plane(k):
        ax = fig.add_axes([(x0 + k * (sw + gap)) / W_, y0 / H_, sw / W_, sh / H_])
        ax.set_xlim(*XL); ax.set_ylim(*YL); ax.set_xticks([]); ax.set_yticks([])
        for s_ in ('left', 'right'):
            ax.spines[s_].set_visible(False)
        for s_ in ('top', 'bottom'):
            ax.spines[s_].set_visible(True); ax.spines[s_].set_linewidth(0.7); ax.spines[s_].set_color('k')
        ax.patch.set_visible(False)
        cx = (x0 + k * (sw + gap) + sw / 2) / W_
        fig.text(cx, (y0 + sh + 0.03) / H_, titles[k], fontsize=6.8, color='k', ha='center', va='bottom')
        fig.text(cx, (y0 - 0.13) / H_, rules[k], fontsize=5.8, color='k', ha='center', va='center')
        return ax

    def circ(ax, x, y, r_in, **kw):
        ax.add_patch(mpl.patches.Ellipse((x, y), 2 * r_in * xpi, 2 * r_in * ypi, **kw))

    def glyph(ax, P, name, ghost=False):
        x, y = P
        if ghost:
            circ(ax, x, y, 0.085, fc='none', ec=GREEN, lw=0.85, ls=(0, (2.2, 1.4)), zorder=2)
            return
        circ(ax, x, y, 0.085, fc='white', ec=NEUTRAL, lw=0.8, zorder=3)
        circ(ax, x, y, 0.024 + 0.014 * y, fc=GOLD, ec='none', zorder=4)
        right = name == 'j'
        ax.text(x + (1 if right else -1) * (R_ + 0.04), y + R_ * 0.55, SHOW[name], fontsize=7.2, style='italic',
                color='k', ha='left' if right else 'right', va='bottom', zorder=6)

    def links(ax, S, arc_ij=False):
        for a, b in LINKS:
            if arc_ij and (a, b) == ('i', 'j'):
                ax.add_patch(FancyArrowPatch(S[a], S[b], connectionstyle='arc3,rad=-0.45', arrowstyle='-',
                                             color=BLUE, lw=1.4, alpha=0.55, zorder=1))
            else:
                ax.plot(*zip(S[a], S[b]), color=BLUE, lw=1.4, alpha=0.55, solid_capstyle='round', zorder=1)

    def axis_labels(ax):
        ax.text(-0.72, YL[1] - 0.06, r'payoff  $c$', fontsize=5.6, color='#8C6D0E', ha='left', va='top')
        ax.text(2.12, YL[0] + 0.06, r'strategy  $\beta$', fontsize=5.6, color=GREEN, ha='right', va='bottom',
                zorder=6, bbox=dict(boxstyle='square,pad=0.08', fc='white', ec='none'))

    # 1. exploration spreads the strategies from one start, while every element still earns alike
    ax = plane(0); axis_labels(ax)
    bs = 1.02
    ax.plot([bs, bs], [YL[0], YL[1]], color=GREY, lw=0.7, ls=(0, (1.6, 1.6)), zorder=0)
    ax.text(bs + 0.06, YL[1] - 0.06, 'start', fontsize=5.4, color=GREY, ha='left', va='top')
    links(ax, S1, arc_ij=True)
    for n in 'ikj':
        glyph(ax, S1[n], n)
    ya = cM - R_ - 0.16
    for n in ('i', 'j'):
        ax.add_patch(FancyArrowPatch((bs, ya), (S1[n][0], ya), arrowstyle='-|>', mutation_scale=6,
                                     color=GREEN, lw=1.0, zorder=5))
    ax.text(bs, ya - 0.10, 'explore', fontsize=5.8, color=GREEN, ha='center', va='top', zorder=6)

    # 2. u stimulates k, and k pays each partner through that partner's own allocation: j, directed to k by
    #    its larger beta, gathers more and rises to the capability of k
    ax = plane(1)
    links(ax, S2)
    for n in 'ikj':
        glyph(ax, S2[n], n)
    wavy(ax, (XL[0] + 0.06, cH), (bL - R_ - 0.05, cH), amp=0.045, waves=2.5, side='above', fs=6.8)
    ax.add_patch(FancyArrowPatch(S2['k'], S2['j'], arrowstyle='-|>', mutation_scale=8, color=GOLD, lw=2.2,
                                 shrinkA=8.5, shrinkB=8.5, zorder=5))
    ax.add_patch(FancyArrowPatch(S2['k'], S2['i'], arrowstyle='-|>', mutation_scale=5, color=GOLD, lw=0.8,
                                 shrinkA=8.5, shrinkB=8.5, zorder=5))
    ax.text(2.10, YL[0] + 0.08, 'payoff rises\nwith strategy', fontsize=5.6, color='#8C6D0E', ha='right',
            va='bottom', linespacing=0.95)

    # 3. all at once, the worse-paid i copies the better-paid j, and the mean strategy moves.
    #    i copying k would change nothing, since k holds the strategy of i.
    ax = plane(2)
    links(ax, S2)
    for n in 'ikj':
        glyph(ax, S2[n], n)
    G_ = (bR, cL)
    glyph(ax, G_, None, ghost=True)
    ax.add_patch(FancyArrowPatch(S2['j'], G_, arrowstyle='-|>', mutation_scale=7, color=RED, lw=1.25,
                                 shrinkA=8.5, shrinkB=8.5, zorder=5))
    ax.text(bR + 0.13, (cH + cL) / 2, r'copy $\beta_j$', fontsize=5.8, color=RED, ha='left', va='center',
            rotation=90)
    ax.add_patch(FancyArrowPatch(S2['i'], G_, arrowstyle='-|>', mutation_scale=6.5, color=GREEN, lw=1.1,
                                 shrinkA=8.5, shrinkB=8.5, zorder=5))
    b0 = (bL + bL + bR) / 3; b1 = (bR + bL + bR) / 3
    yb = YL[0] + 0.13
    for b, c_ in ((b0, GREY), (b1, GREEN)):
        ax.plot([b, b], [YL[0], YL[0] + 0.15], color=c_, lw=1.2, zorder=6)
    ax.annotate('', xy=(b1, yb + 0.12), xytext=(b0, yb + 0.12),
                arrowprops=dict(arrowstyle='-|>', color=GREEN, lw=1.0, mutation_scale=6))
    ax.text(b0 - 0.06, yb + 0.12, r'$\Delta\bar{\beta} > 0$', fontsize=5.8, color=GREEN, ha='right', va='center')

    for k in range(2):                                     # stage arrows
        xa = x0 + (k + 1) * sw + k * gap
        fig.add_artist(FancyArrowPatch(((xa + 0.03) / W_, (y0 + sh / 2) / H_), ((xa + gap - 0.03) / W_, (y0 + sh / 2) / H_),
                                       transform=fig.transFigure, arrowstyle='-|>', mutation_scale=6,
                                       color=NEUTRAL, lw=0.8))
    fig.text((ox + pw / 2) / W_, (oy + 0.13) / H_,
             r'$E[\Delta\bar{\beta}] = \frac{p_r}{N}\sum_i \frac{1}{k_i}\sum_{j \in \mathcal{N}_i}\varphi(c_j - c_i)(\beta_j - \beta_i)$'
             r'$\qquad\mathrm{(Eq.\ 15)}$', fontsize=6.4, color='k', ha='center', va='center')


def mark(ax, y, color=RED):
    for t, L in zip(MOMENTS, 'AC'):
        i = int(np.argmin(np.abs(T - t)))
        ax.plot([T[i]], [y[i]], 'o', ms=3.4, mfc='white', mec=color, mew=0.8, zorder=5, clip_on=False)


def draw_D(fig, ox, oy, letter):
    pred = np.array([r['pred'] for r in RUNS['both']])
    bm = stack('both', 'beta_mean')
    cum = np.c_[np.zeros(len(pred)), pred.cumsum(1)][:, ::EVERY]       # Eq. 15 accumulated, at the recorded steps
    ax = axes_at(fig, ox, oy, BOX_2)
    frame(ax, XT, [0, 1.5, 3])
    ax.plot(T, bm.mean(0), color=GREEN, lw=1.2, zorder=3)
    ax.plot(T, cum.mean(0), color=RED, lw=1.0, ls=(0, (3.2, 2.0)), zorder=4)
    mark(ax, bm.mean(0), GREEN)
    ax.set_xlabel('step', labelpad=1.6)
    ax.set_ylabel(r'direction preference  $\bar{\beta}$', labelpad=2.0)
    var = stack('both', 'beta_var')
    # the variance across elements on the same axis, ten times enlarged, so the panel needs no second axis
    ax.fill_between(T, 0, 10 * np.median(var, 0), color=GREEN, alpha=0.20, lw=0, zorder=1)
    hs = [mpl.patches.Patch(color=GREEN, alpha=0.25, lw=0), Line2D([], [], color=RED, lw=1.0, ls=(0, (3.2, 2.0))),
          Line2D([], [], color=GREEN, lw=1.2)]
    # the longest label in the top row, where the curves never reach
    ax.legend(hs, [r'variance of $\beta_i$, $\times$10', 'Eq. 15, summed', r'$\bar{\beta}$, measured'],
              loc='upper left', frameon=False, handlelength=1.5, borderaxespad=0.2, labelspacing=0.22)
    heading(fig, ox, oy, PW_B, PH_B, letter, 'Variation turns into direction')


def draw_E(fig, ox, oy, letter):
    '''The degree distribution at the end of the runs, the thirty runs of each condition pooled, drawn as
    Fig. 2F draws the four structures. The lattice is the single step at six.'''
    ax = axes_at(fig, ox, oy, BOX_2)
    ax.set_xscale('log'); ax.set_yscale('log')
    frame(ax, [1, 10, 100], [0.001, 0.01, 0.1, 1])
    ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f'{v:g}'))
    ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f'{v:g}'))
    LAT = dict(color=GREEN, lw=1.0, ls=(0, (1.0, 1.6)), zorder=3)
    ax.plot([6, 6], [1, 1e-3], **LAT)
    ax.plot([1, 6], [1, 1], **LAT)
    grid = np.unique(np.round(np.logspace(np.log10(2), 2, 26)).astype(int))
    OTHER = '#8A8A8A'
    for n, ls in (('baseline', '-'), ('imitate', ':'), ('explore', (0, (3, 2)))):
        dd = np.concatenate([(r['snaps'][TMAX]['W'] > 0).sum(1) for r in RUNS[n]])
        ks = np.arange(1, dd.max() + 1)
        ax.plot(ks, [(dd >= k).mean() for k in ks], color=OTHER, lw=0.85, ls=ls, zorder=3)
    dd = np.concatenate([(r['snaps'][TMAX]['W'] > 0).sum(1) for r in RUNS['both']])
    ks = np.arange(1, dd.max() + 1); cc = np.array([(dd >= k).mean() for k in ks])
    ax.plot(ks, cc, '-', color=NAVY, lw=1.1, zorder=4)
    g = grid[grid <= dd.max()]
    ax.plot(g, cc[g - 1], ls='none', marker='^', ms=3.0, mfc=NAVY, mec='white', mew=0.35, zorder=5)
    ax.set_xlabel(r'degree  $k$', labelpad=1.6)
    ax.set_ylabel(r'$P(\geq k)$', labelpad=2.0)
    ax.text(0.97, 0.95, f'step {TMAX:,}', transform=ax.transAxes, fontsize=5.8, ha='right', va='top')
    ax.text(95, 0.05, 'exploration\nand imitation', fontsize=5.8, color=NAVY, ha='right', va='bottom',
            linespacing=0.95)
    ax.text(21, 0.0042, 'the other\nthree', fontsize=5.8, color='#6E6E6E', ha='right', va='center',
            linespacing=0.95)
    ax.text(5.6, 0.03, 'lattice', fontsize=5.8, color=GREEN, ha='right', va='center')
    heading(fig, ox, oy, PW_B, PH_B, letter, 'Direction builds hubs')


def draw_F(fig, ox, oy, letter):
    '''Bits saved on degree shares. The combined condition in its own color, the other three pooled, since
    their curves coincide.'''
    ax = axes_at(fig, ox, oy, BOX_2)
    frame(ax, XT, [0, 0.5, 1])
    y = {n: CEIL - stack(n, 'H_deg') for n in ('baseline', 'explore', 'imitate', 'both')}
    rest = np.vstack([y['baseline'], y['explore'], y['imitate']])
    ax.fill_between(T, np.percentile(rest, 25, 0), np.percentile(rest, 75, 0), color=GREY, alpha=0.28, lw=0,
                    zorder=2)
    ax.plot(T, np.median(rest, 0), color='#6E6E6E', lw=0.9, zorder=3)
    ax.fill_between(T, np.percentile(y['both'], 25, 0), np.percentile(y['both'], 75, 0), color=NAVY,
                    alpha=0.16, lw=0, zorder=2)
    med = np.median(y['both'], 0)
    ax.plot(T, med, color=NAVY, lw=1.25, zorder=4)
    mark(ax, med, NAVY)
    ax.set_xlabel('step', labelpad=1.6)
    ax.set_ylabel(r'bits saved  $\log_2 N - H$', labelpad=2.0)
    ax.text(0.98, 0.05, 'degree shares', transform=ax.transAxes, fontsize=5.8, ha='right', va='bottom')
    hs = [(mpl.patches.Patch(color=NAVY, alpha=0.16, lw=0), Line2D([], [], color=NAVY, lw=1.25)),
          (mpl.patches.Patch(color=GREY, alpha=0.28, lw=0), Line2D([], [], color='#6E6E6E', lw=0.9))]
    ax.legend(hs, ['exploration and imitation', 'the other three'], loc='upper left', frameon=False,
              bbox_to_anchor=(0.0, 1.0),
              handlelength=1.8, borderaxespad=0.25, labelspacing=0.22)
    heading(fig, ox, oy, PW_B, PH_B, letter, 'Hubs save bits')


def figure5(label, show=True):
    H_FIG = PH_T + GAP + PH_B
    fig = plt.figure(figsize=(FIG_W, H_FIG)); fig.set_label(label)
    oy = PH_B + GAP
    draw_net(fig, 0, oy, 0, 'A', 'The lattice', 'a ring, even coupling')
    draw_game(fig, PW_T, oy, 'B', 2 * PW_T)
    draw_net(fig, 3 * PW_T, oy, TMAX, 'C', f'Step {TMAX:,}', 'hubs drawn inward by degree')
    W_, H_ = fig.get_size_inches()
    for x0 in (PW_T, 3 * PW_T):
        fig.add_artist(FancyArrowPatch(((x0 - 0.10) / W_, (oy + 1.12) / H_), ((x0 + 0.06) / W_, (oy + 1.12) / H_),
                                       transform=fig.transFigure, arrowstyle='-|>', mutation_scale=7,
                                       color=NEUTRAL, lw=0.9))
    draw_D(fig, 0, 0, 'D')
    draw_E(fig, PW_B, 0, 'E')
    draw_F(fig, 2 * PW_B, 0, 'F')
    if not show:
        return fig
    plt.show()

figure5('Manuscript Fig 5')
figure5('Manuscript Fig 5A-F')

# each panel as its own file, cut from one more drawing of the same figure, as Figs. 2 to 4 are saved panel by
# panel: above, the lattice, the game with the arrows on both sides and the structure at the last step, and
# below, the three plots
fig = figure5('Fig 5 for the panel files', show=False)
TOP = PH_B + GAP
ARROWS = [a for a in fig.artists if isinstance(a, FancyArrowPatch)]   # the arrows that join the panels
LETTER_C = [t for t in fig.texts if t.get_text() == 'C']
save_panels(fig, 'Manuscript_Fig_5', {'A': (0.0, TOP, PW_T - 0.11, TOP + PH_T),
                                      'D': (0.0, 0.0, PW_B, PH_B),
                                      'E': (PW_B, 0.0, 2 * PW_B, PH_B),
                                      'F': (2 * PW_B, 0.0, FIG_W, PH_B)})
save_panels(fig, 'Manuscript_Fig_5', {'B': (PW_T - 0.11, TOP, 3 * PW_T + 0.07, TOP + PH_T)}, hide=LETTER_C)
save_panels(fig, 'Manuscript_Fig_5', {'C': (3 * PW_T - 0.02, TOP, FIG_W, TOP + PH_T)}, hide=ARROWS)
plt.close(fig)

panel files: Manuscript_Fig_5A, Manuscript_Fig_5D, Manuscript_Fig_5E, Manuscript_Fig_5F
panel files: Manuscript_Fig_5B
panel files: Manuscript_Fig_5C


## Step 5. Values for the manuscript, and checks

In [9]:
fmt = lambda v, f: format(v, f).replace('-', '\u2212')
OUT = {
    'F5_T': f'{TMAX:,}', 'F5_SEED': str(REP_VAL['seed']), 'F5_KAPPA_REP': f'{REP_VAL["kappa"]:.2f}',
    'F5_MAXDEG_REP': str(REP_VAL['maxdeg']), 'F5_BETA_REP': fmt(REP_VAL['beta'], '+.2f'),
    'F5_DHDEG_REP': f'{REP_VAL["dH1"]:.2f}', 'F5_TOP5_REP': str(REP_VAL['top5']),
    'F5_EQ15': f'{P_SUM.mean():.2f}', 'F5_BETA_MEAS': f'{M_SUM.mean():.2f}', 'F5_EQ15_DIFF': fmt(DIFF.mean(), '+.2f'),
    'F5_EQ15_SE': f'{SE:.2f}', 'F5_T_BETA05': f'{t05:,}' if t05 else 'none',
    'F5_T_F001': f'{t_f:,}' if t_f else 'none',
    'F5_DHDEG_BOTH': f'{np.median(END["both"]["dH_deg"]):.2f}', 'F5_DHDEG_REST': f'{np.median(rest("dH_deg")):.2f}',
    'F5_MAXDEG_BOTH': f'{np.median(END["both"]["maxdeg"]):.0f}', 'F5_MAXDEG_REST': f'{np.median(rest("maxdeg")):.0f}',
    'F5_KMAX_BOTH': str(int(pool['both'].max())),
    'F5_KMAX_REST': str(int(max(pool[n].max() for n in ('baseline', 'explore', 'imitate')))),
}
pfmt = lambda p: 'P < 0.001' if p < 0.001 else f'P = {p:.3f}'
for name, key in (('baseline', 'BASE'), ('explore', 'EXP'), ('imitate', 'IMI'), ('both', 'BOTH')):
    OUT[f'F5_DHDEG_{key}_CI'] = f'[{DHDEG_CI[name][0]:.3f},_{DHDEG_CI[name][1]:.3f}]'   # no spaces in a value line
for t, key in ((T_DEG_EXP, 'EXP'), (T_DEG_BASE, 'BASE')):
    OUT[f'F5_TDEG_{key}_P'] = pfmt(t['p']).replace(' ', '_')
    OUT[f'F5_TDEG_{key}_D'] = fmt(t['d'], '+.2f')
    OUT[f'F5_TDEG_{key}_U'] = f"{t['U']:.1f}"
    OUT[f'F5_TDEG_{key}_DIFF'] = fmt(t['diff'], '+.3f')
    OUT[f'F5_TDEG_{key}_CI'] = '[' + fmt(t['lo'], '+.3f') + ',_' + fmt(t['hi'], '+.3f') + ']'
for k, v in OUT.items():
    print(f'{k} = {v}')
print('check, a followed run equals the same seed run straight through:', CHECK_PLAIN)
print('check, Eq. 15 summed within two standard errors of the measured change:', CHECK_EQ15)

F5_T = 5,000
F5_SEED = 13
F5_KAPPA_REP = 21.05
F5_MAXDEG_REP = 86
F5_BETA_REP = +2.54
F5_DHDEG_REP = 0.73
F5_TOP5_REP = 5
F5_EQ15 = 1.96
F5_BETA_MEAS = 1.74
F5_EQ15_DIFF = +0.22
F5_EQ15_SE = 0.18
F5_T_BETA05 = 1,325
F5_T_F001 = 350
F5_DHDEG_BOTH = 0.71
F5_DHDEG_REST = 0.38
F5_MAXDEG_BOTH = 84
F5_MAXDEG_REST = 25
F5_KMAX_BOTH = 99
F5_KMAX_REST = 37
F5_DHDEG_BASE_CI = [0.363,_0.377]
F5_DHDEG_EXP_CI = [0.384,_0.420]
F5_DHDEG_IMI_CI = [0.362,_0.382]
F5_DHDEG_BOTH_CI = [0.591,_0.768]
F5_TDEG_EXP_P = P_<_0.001
F5_TDEG_EXP_D = +0.63
F5_TDEG_EXP_U = 733.0
F5_TDEG_EXP_DIFF = +0.307
F5_TDEG_EXP_CI = [+0.186,_+0.368]
F5_TDEG_BASE_P = P_<_0.001
F5_TDEG_BASE_D = +0.65
F5_TDEG_BASE_U = 743.0
F5_TDEG_BASE_DIFF = +0.342
F5_TDEG_BASE_CI = [+0.223,_+0.402]
check, a followed run equals the same seed run straight through: True
check, Eq. 15 summed within two standard errors of the measured change: True
